# NWT version 2 (august 2019) data 

Read NWT monitoring data and parse it into a usable format
Print with either "below LOD", "LOD" or "Nan" values for observatations below LOD

<Hr>
    
#### Import libraries

In [66]:
import pandas as pd
from pandas import ExcelWriter
from pandas import ExcelFile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import datetime
from matplotlib.backends.backend_pdf import PdfPages

#### Read data

In [67]:
# read file
path='/Users/als/Dropbox/Mackenzie Hg project/Supporting data/NWT Community-based water quality monitoring program/'

# file to read
fname=path+'NWT-wide Community-based Monitoring Program_data.csv'
df = pd.read_csv(fname, low_memory=False)
dft = df.set_index('LaboratoryName').drop('Field Data',axis=0)
df = dft.reset_index()

df = df.drop(['DatasetName','MonitoringLocationID','MonitoringLocationWaterbody','ActivityEndDate','ActivityEndTime',
             #'ResultSampleFraction',#'ResultDetectionCondition','ResultDetectionQuantitationLimitMeasure',
             'ResultDetectionQuantitationLimitType',#'ResultDetectionQuantitationLimitUnit',
             'ResultStatusID','ResultAnalyticalMethodID','ResultAnalyticalMethodContext',#'ResultComment',
             'ResultAnalyticalMethodName','AnalysisStartDate','AnalysisStartTime','AnalysisStartTimeZone',
             'LaboratoryName','LaboratorySampleID','MethodSpeciation','ActivityStartTime',
             'MonitoringLocationHorizontalCoordinateReferenceSystem','ActivityType','ActivityMediaName',
             'SampleCollectionEquipmentName'],axis=1)

df=df.rename(index=str, columns={"MonitoringLocationName":"Location","MonitoringLocationLatitude":"Latitude",
                                "MonitoringLocationLongitude":"Longitude","MonitoringLocationType":"Waterbody type",
                                "CharacteristicName":"Sample_type","ActivityStartDate":"Date",#'ActivityStartTime':'Time',
                                 "ResultDetectionCondition":"Below_detection","ResultDetectionQuantitationLimitMeasure":"LOD",
                                "ActivityDepthHeightMeasure":"Depth","ActivityDepthHeightUnit":"Depth_unit",
                                "ResultDetectionQuantitationLimitUnit":"LOD_unit"})

# convert date to Datetime
df["Date"]=pd.to_datetime(df.Date)

# fill nan for Time column
#values = {'Time': '00:00:00'}
values = {'ResultComment': 'None'}
df = df.fillna(value=values)

df['Depth'] = df['Depth']*(-1)
values = {'Depth': 0, 'Depth_unit': 'm'}
df = df.fillna(value=values)

# extract only River/stream water bodies
df1 = df.set_index('Waterbody type')

# Mark if focus is on only rivers or lakes
df1 = df1.loc['River/Stream'] 
#df1 = df1.loc['Lake/Pond']

df1 = df1.reset_index().drop('Waterbody type', axis=1)

# drop unwanted elements
df1 = df1.set_index('Sample_type').drop(['Antimony','Bismuth','Beryllium','Boron','Cesium','Cobalt','Chromium','Molybdenum','Orthophosphate',
                                        'Uranium','True color','Silica, reactive','Silver','Chlorophyll a (probe)','Fecal Coliform','Selenium',
                                        'Ethylbenzene','Streptococcus','Toluene','Total Coliform','Escherichia coli','Enterococcus','Thallium',
                                        'Benzene','Oxidation reduction potential (ORP)','Dissolved oxygen (DO)    ','Temperature, water ',
                                        'Nitrite','Nitrate','Xylene','Tin','Hydrocarbons','Inorganic nitrogen (nitrate and nitrite)',
                                        'Dissolved oxygen saturation'], axis=0)

df1 = df1.reset_index()

# Create unique names for variables - split and concat again
df1notnan = df1.dropna(subset=['ResultSampleFraction']).copy(deep=True)
df1notnan['Sample_type'] = df1notnan['Sample_type'] + df1notnan['ResultSampleFraction'] #A value is trying to be set on a copy of a slice from a DataFrame.

df1isnull = df1[pd.isnull(df1['ResultSampleFraction'])]

df1a=pd.concat([df1notnan,df1isnull]).drop('ResultSampleFraction',axis=1)#, sort=True)

# Remove non-trace metal Hg samples (ug/L)
df1b = df1a.set_index(['Sample_type'])
df1b_m = df1b.loc[['MercuryTotal','MercuryDissolved'],:]

df1b_m = df1b_m.reset_index()
df1b_m = df1b_m.set_index(['Sample_type','ResultUnit'])
df1b_m = df1b_m.drop('ug/L',level=1)
df1b_m = df1b_m.reset_index()
df1b_m = df1b_m.set_index(['Sample_type','LOD_unit'])
df1b_m = df1b_m.drop('ug/L',level=1)
df1b_m = df1b_m.reset_index()

df1b_r = df1b.drop(['MercuryTotal','MercuryDissolved'], axis=0)
df1b_r = df1b_r.reset_index()

df1c=pd.concat([df1b_m,df1b_r], sort=True).drop('LOD_unit',axis=1)

#### Make sure each variable only have one unit represented
Uncomment for use when adding new variables that needs to be checked

In [68]:
#df_doub = df1c.set_index(['Sample_type','ResultUnit'])
#df_doub = df_doub.groupby(level = ['Sample_type','ResultUnit']).mean()
#df_doub

#### Create dataset with unique Location, dates and depths for where mercury data present
 Output used as basis for the dataset found in our shared folder
 
 Choose option in output for values below LOD: '<LOD','LOD','Nan' (in this last case nothing is done)

In [69]:
# Set output, options: '<LOD','LOD','Nan' (in this last case nothing is done)
######################
LOD = 'Nan'
######################

df1d = df1c
df_loc0 = df1d.drop(['Depth_unit','ResultUnit','ResultValue','Below_detection','LOD'],axis=1)#'Sample_type',
df_loc0 = df_loc0.set_index('Sample_type').loc[['MercuryTotal','MercuryDissolved'],:]
df_loc0 = df_loc0.reset_index().drop('Sample_type',axis=1)
df_loc0 = df_loc0.drop_duplicates(subset=None, keep='first')
df_loc  = df_loc0

# create lists with variable names in original spreadsheet and new names for new spreadsheet (that includes units)
df_variable=df1d.drop(['Location','Latitude','Longitude','Date','Depth','Depth_unit','ResultValue','Below_detection','LOD'],axis=1)
df_variable = df_variable.drop_duplicates(subset=['Sample_type'])
df_variable['Name']=df_variable['Sample_type']+' ('+df_variable['ResultUnit']+')'
df_variable['Name']=df_variable['Name'].fillna(df_variable['Sample_type'])

List=df_variable['Sample_type'].tolist()
List_name=df_variable['Name'].tolist()

# loop over all variables and merge with unique set of locations, depths and times
df2=df1d.set_index(['Sample_type']).drop(['Latitude','Longitude','Depth_unit','ResultUnit'],axis=1)
n=len(List)

for id in range(n):
    Alk_t = df2.loc[[List[id]],:]
    Alk_t=Alk_t.rename(index=str, columns={"ResultValue":List_name[id]})
    Alk_t = Alk_t.reset_index().drop('Sample_type',axis=1)

    # Extracts all rows for the given variable that is not Nan
    df2notnan = Alk_t.dropna(subset=[List_name[id]])
    
    # Extracts all rows for the given variable that is Nan/missing value and inserts "<LOD" in this field
    df2isnull = Alk_t[pd.isnull(Alk_t[List_name[id]])].copy(deep=True)

    if LOD == '<LOD':
        df2isnull[List_name[id]] = '<'+df2isnull['LOD'].astype(str) #A value is trying to be set on a copy of a slice from a DataFrame.
    if LOD == 'LOD':
        df2isnull[List_name[id]] = df2isnull['LOD']
        
    # The two datasets are concatenated into one
    Alk_t1=pd.concat([df2notnan,df2isnull]).drop(['LOD','Below_detection'],axis=1)
        
    df_loc=pd.merge(df_loc,Alk_t1,how='left',left_on=['Location','Date','Depth','ResultComment'],right_on=['Location','Date','Depth','ResultComment'])

df_loc['Year'] = pd.DatetimeIndex(df_loc['Date']).year
df_loc['Month'] = pd.DatetimeIndex(df_loc['Date']).month
df_loc['Day'] = pd.DatetimeIndex(df_loc['Date']).day

#### Print to excel
Version with "<LOD","LOD" or "Nan"

In [70]:
if LOD == '<LOD':
    df_loc.to_excel(path+'NWT_overview_RIVERS_STREAMS_to_2018.xlsx', sheet_name='Sheet1')
if LOD == 'LOD':
    df_loc.to_excel(path+'NWT_overview_RIVERS_STREAMS_to_2018_LODeqLOD.xlsx', sheet_name='Sheet1')
if LOD == 'Nan':
    df_loc.to_excel(path+'NWT_overview_RIVERS_STREAMS_to_2018_LODeqNan.xlsx', sheet_name='Sheet1')